In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## IMPORT LIBRARIES AND DATASETS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from sklearn import datasets

data = datasets.load_iris()
dir(data)

In [ ]:
df = pd.DataFrame(data.data, columns = data.feature_names)
df['target'] = data.target

df.head()

In [ ]:
X = df.drop(columns = ['target'])
y = df['target']

X.shape, y.shape

## DATAPROCESSING

In [ ]:
from sklearn.preprocessing import MinMaxScaler

s = MinMaxScaler()
X[:] = s.fit_transform(X)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

X_train.shape, y_train.shape

In [ ]:
from sklearn.svm import SVC

model1 = SVC(kernel = 'rbf', C = 30, gamma = 'auto')
model1.fit(X_train, y_train)
model1.score(X_test, y_test)

## HYPER PARAMETER TUNING - K-FOLD VALIDATION

In [ ]:
from sklearn.model_selection import cross_val_score

cross_val_score(SVC(kernel = 'linear', C = 10, gamma= 'auto'), X, y, cv = 10)

In [ ]:
cross_val_score(SVC(kernel = 'rbf', C = 10, gamma= 'auto'), X, y, cv = 10)

In [ ]:
kernels = ['rbf', 'linear']
C = [1,10,20]
avg_scores = {}
for kval in kernels:
    for cval in C:
        cv_scores = cross_val_score(SVC(kernel=kval,C=cval,gamma='auto'),X, y, cv=10)
        avg_scores[kval + '_' + str(cval)] = np.average(cv_scores)

avg_scores

## GRID SEARCH TO FIND BEST MODEL

In [ ]:
from sklearn.model_selection import GridSearchCV

clf = GridSearchCV(SVC(gamma = 'auto'), {
    'C': [1, 10, 20],
    'kernel': ['rbf','linear']
}, cv = 10, return_train_score = False)

clf.fit(X, y)
clf.cv_results_

In [ ]:
df = pd.DataFrame(clf.cv_results_)
df

In [ ]:
df[['param_C','param_kernel','mean_test_score', 'rank_test_score']]

In [ ]:
dir(clf)

In [ ]:
clf.best_score_

In [ ]:
clf.best_params_

In [ ]:
# The gridsearch takes all the combination, which directly increases the computation power of the machine, and it drains energy, so we use randomised search strategy

from sklearn.model_selection import RandomizedSearchCV

rs = RandomizedSearchCV(SVC(gamma='auto'), {
        'C': [1,10,20],
        'kernel': ['rbf','linear']
    }, 
    cv=5, 
    return_train_score=False, 
    n_iter=5
)

rs.fit(X, y)
pd.DataFrame(rs.cv_results_)[['param_C','param_kernel','mean_test_score', 'rank_test_score']]

In [ ]:
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

model_params = {
    'svm': {
        'model': svm.SVC(gamma='auto'),
        'params' : {
            'C': [1,10,20],
            'kernel': ['rbf','linear']
        }  
    },
    'random_forest': {
        'model': RandomForestClassifier(),
        'params' : {
            'n_estimators': [1,5,10]
        }
    },
    'logistic_regression' : {
        'model': LogisticRegression(solver='liblinear',multi_class='auto'),
        'params': {
            'C': [1,5,10]
        }
    }
}

In [ ]:
scores = []

for model_name, mp in model_params.items():
    clf =  GridSearchCV(mp['model'], mp['params'], cv=5, return_train_score=False)
    clf.fit(X, y)
    scores.append({
        'model': model_name,
        'best_score': clf.best_score_,
        'best_params': clf.best_params_
    })
    
df = pd.DataFrame(scores,columns=['model','best_score','best_params'])
df